[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/MIMO_Communications.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# MIMO Communications

Multiple antennas at both ends turn multipath — [Digital Comms'](./Digital_Communications.ipynb) villain — into **extra spectrum out of thin air**: parallel spatial channels through the same band. Capacity with many antennas, diversity vs multiplexing, and SVD precoding that turns the channel into independent pipes (verified: the pipes really are independent).

## 1. Pre-requisites

[Digital Communications](./Digital_Communications.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S4 (SVD — the star of Session 3), [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def qpsk_syms(n): return (rng.choice([-1,1],n) + 1j*rng.choice([-1,1],n))/np.sqrt(2)
def rayleigh(nr, nt): return (rng.standard_normal((nr,nt)) + 1j*rng.standard_normal((nr,nt)))/np.sqrt(2)

---
### 🕐 Session 1 of 3 — *MIMO Capacity* (~35 min)
**Goal:** why capacity grows LINEARLY with min(antennas): the log-det formula, simulated.
**Builds on:** [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4. &nbsp; **Feeds into:** Session 2 (diversity).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: MIMO Capacity</b></summary>

**Timing (~35 min).** 8 min the villain-becomes-resource reframe · 10 min the log-det formula · 10 min the simulation · 7 min why the scaling is not quite linear.

**Open with the reversal, because it is the hook of the whole workshop.** In [Digital Communications](./Digital_Communications.ipynb), multipath was the enemy: delayed copies smearing symbols, requiring equalisation. Here it is the *resource*. Rich scattering is what makes $H$ full rank, and rank is what creates parallel spatial channels. Say the consequence out loud — a MIMO link in an open field with a clean line of sight performs *worse* than one in a cluttered room, because line-of-sight gives a rank-1 channel and the pipes collapse. Students find that genuinely counterintuitive, which is why it sticks.

**Build the log-det formula from the SVD rather than quoting it.** $H = U\Sigma V^H$ has $\min(n_t, n_r)$ singular values, each an independent spatial direction with its own gain. Capacity is then the sum of $\log_2(1 + \text{SNR}_i)$ over those pipes, which is exactly what $\log_2\det(I + \frac{\rho}{n_t}HH^H)$ computes. Framing capacity as "add up the pipes" makes Session 3's precoding feel like the obvious next step rather than a new trick.

**Ask the room before running.** "SISO capacity is $\log_2(1+\rho)$, so at 20 dB about 6.6 bits/s/Hz. Doubling the SNR adds *one* bit. How many antennas would you need to double the rate?" Two. Capacity is logarithmic in power and *linear* in antennas — that asymmetry is the entire economic argument for MIMO, and it is why 5G and Wi-Fi add antennas rather than transmit power.

**Handle the 7.4 honestly — it is the interesting number, not a blemish.** $C(8)/C(1) = 7.4$ rather than 8, and the printed comment gives the reason: total power is split across transmit antennas, so each pipe sees $\rho/n_t$ rather than $\rho$. Per-antenna capacity therefore drifts down slightly (5.90 → 5.65 → 5.55 → 5.50 bits/s/Hz) rather than staying fixed. Have the room predict whether the shortfall grows or shrinks with more antennas — it converges, because the loss is a $\log$ of a shrinking factor against a linearly growing count. "Linear scaling" is an asymptotic statement with a constant, not an exact law.

**Note what the simulation is doing.** `np.mean` over 2000 channel draws is **ergodic** capacity — the average over fading realisations, appropriate when a codeword spans many channel states. That is a different quantity from outage capacity, which is what matters for a delay-limited link and is governed by the *bad* draws rather than the average. Session 2 is about exactly those bad draws, so flagging the distinction here sets it up.
</details>

## 2. Spectrum from Space

💡 **Intuition.** A rich-scattering channel matrix $H$ has $\min(n_t, n_r)$ meaningful singular directions — each an independent spatial pipe. Capacity $C = \log_2\det(I + \frac{\rho}{n_t} H H^H)$ therefore grows ~**linearly** in $\min(n_t, n_r)$ at high SNR, while a single antenna only ever gets $\log(1{+}\rho)$. Multipath, the enemy of the single-antenna link, is the *resource* here: no scattering ⇒ rank-1 $H$ ⇒ pipes collapse.

In [2]:
snr_db = 20; rho = 10**(snr_db/10)
antennas = [1, 2, 4, 8]
cap = {n_a: np.mean([np.log2(np.linalg.det(np.eye(n_a) + rho/n_a * (H := rayleigh(n_a, n_a)) @ H.conj().T).real)
                     for _ in range(2000)]) for n_a in antennas}
plt.figure(figsize=(7, 2.8))
plt.plot(antennas, [cap[a] for a in antennas], "o-", label="i.i.d. Rayleigh (rich scattering)")
plt.plot(antennas, [np.log2(1+rho)]*4, "k--", linewidth=1, label="1×1 (SISO) ceiling")
plt.xlabel("antennas (n×n)"); plt.ylabel("bits/s/Hz"); plt.legend()
plt.title(f"@{snr_db} dB: ergodic capacity scales ~linearly with antennas")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("mean capacity [bits/s/Hz]:", {a: round(cap[a],1) for a in antennas})
print(f"slope check: C(8)/C(1) = {cap[8]/cap[1]:.1f}  (linear scaling would give ≈8; the shortfall is the ρ/n_t power split)")

mean capacity [bits/s/Hz]: {1: np.float64(5.9), 2: np.float64(11.3), 4: np.float64(22.2), 8: np.float64(44.0)}
slope check: C(8)/C(1) = 7.4  (linear scaling would give ≈8; the shortfall is the ρ/n_t power split)


/tmp/ipykernel_2992188/166511356.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


**What just happened.** Ergodic capacity at 20 dB: **5.9 / 11.3 / 22.2 / 44.0** bits/s/Hz for 1, 2, 4, and 8 antennas. Roughly a doubling each time the antenna count doubles, against a SISO ceiling that sits flat no matter how many antennas the plot's x-axis claims.

**The asymmetry is the economic argument for MIMO.** Capacity is *logarithmic* in power — doubling transmit power at 20 dB buys you about one extra bit — and *linear* in antennas. So going from 1 to 8 antennas gains 38 bits/s/Hz, while achieving the same by raising power would require an increase of roughly $2^{38}$. That is why 5G and Wi-Fi kept adding antennas rather than transmitters, and it is the single most consequential fact in this workshop.

**Now the 7.4, which is more interesting than a clean 8 would be.** Perfect linear scaling predicts $C(8)/C(1) = 8$; we measured 7.4. The shortfall is real and the printed comment names it: total transmit power is divided across antennas, so each pipe sees $\rho/n_t$ instead of $\rho$. Per-antenna capacity drifts down accordingly — 5.90, 5.65, 5.55, 5.50 bits/s/Hz — because each pipe is individually a little weaker even as there are more of them.

Note the shape of that drift: it is *converging*, not collapsing. The loss per pipe is a logarithm of a shrinking factor while the pipe count grows linearly, so the linear term wins asymptotically. "Capacity scales linearly with antennas" is a statement about the slope at high SNR, with a constant that this simulation makes visible. Precision about which part of a scaling law is exact and which is asymptotic is worth more than the headline.

**Where the rank comes from — and the counterintuitive consequence.** Those pipes exist only because $H$ has $\min(n_t, n_r)$ meaningful singular values, and $H$ is full rank only because the environment scatters richly. A clean line-of-sight channel gives a rank-1 $H$: all the singular values but one collapse, the pipes vanish, and eight antennas buy array gain but no multiplexing. **A MIMO link works better in a cluttered room than in an open field.** Multipath, the villain of [Digital Communications](./Digital_Communications.ipynb), is here the resource being harvested.

**One caveat on what was measured.** Averaging over 2000 channel draws gives *ergodic* capacity — the long-run average across fading states, which is the right quantity when a codeword spans many channel realisations. It says nothing about any individual bad draw. For a latency-limited link the governing quantity is outage capacity, determined by the unlucky realisations rather than the mean, and those bad draws are exactly what Session 2 is about.

---
### 🕐 Session 2 of 3 — *Diversity: Never Fade Alone* (~40 min)
**Goal:** Rayleigh fading murders BER; independent branches resurrect it — the diversity-order slopes.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (SVD precoding).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Diversity — Never Fade Alone</b></summary>

**Timing (~40 min).** 10 min why fading is worse than it looks · 10 min the diversity argument · 12 min the demo and reading slopes · 8 min MRC and Alamouti.

**Board first — the counterintuitive fact about fading.** On a Rayleigh channel the *average* SNR can be perfectly healthy while the link is terrible, because BER is dominated by the probability of being in a deep fade, and that probability falls only as $1/\rho$. Contrast with AWGN, where BER falls exponentially in SNR. Ask the room what that means practically: on a fading channel, adding 10 dB of transmit power buys you one order of magnitude of BER, whereas on AWGN it buys several. Power is a bad way to fight fading.

**Then the resource that actually works.** With $L$ **independent** branches, a deep fade requires *all* of them to fade simultaneously — probability $\rho^{-L}$. So the BER-versus-SNR *slope* on a log-log plot becomes $L$. Insist on the word slope: diversity does not shift the curve down, it *tilts* it, and the benefit therefore grows without bound as SNR rises. That distinction is the session.

**Teach students to read diversity order off a plot.** On log-BER against dB-SNR, count decades of BER per 10 dB. Slope 1 means one decade per 10 dB; slope 4 means four. The dotted reference line is there for exactly this. Have the room measure each curve's slope from the figure rather than being told — it is a skill that transfers to every fading plot they will ever see.

**Emphasise "independent".** Diversity order $L$ requires the branches to fade independently. Antennas spaced much less than half a wavelength are strongly correlated and deliver far less than $L$; the practical rules of thumb about antenna spacing exist entirely for this reason. Ask what happens with two perfectly correlated branches — you get array gain (a shift) but no extra slope, since they always fade together.

**MRC is a matched filter, and the room already owns it.** `(h.conj() * y).sum(0)` weights each branch by its own conjugate channel — strong branches count more, weak ones are down-weighted rather than discarded. That is precisely the [matched filter](./Statistical_Signal_Processing.ipynb) argument applied across antennas rather than across time, and it is provably the optimal linear combiner in white noise. One line of code, one theorem.

**Mention Alamouti even though it is not implemented.** MRC needs multiple *receive* antennas and channel knowledge at the receiver. Alamouti's space-time block code gets full transmit diversity from two transmit antennas with **no channel knowledge at the transmitter**, using a clever orthogonal structure across two symbol periods. That is why it is in every cellular standard — handsets can have one antenna and still benefit. If a student asks what makes it famous, that is the answer.

**If the demo is slow.** 200,000 bits at each of 6 SNR points for 3 diversity orders is a few seconds. At the highest SNR and $L = 4$ the BER may hit zero errors, which vanishes from a log plot — if a point disappears, that is why, and more bits is the fix.
</details>

## 3. Insurance Against Fades

💡 **Intuition.** On a fading channel the *average* SNR is fine; the *bad moments* kill you — BER is dominated by the probability the channel is in a deep fade, which falls only as $1/\rho$ (diversity order 1). With $L$ **independent** branches, all must fade together: $P \sim \rho^{-L}$ — the BER-vs-SNR slope steepens to $L$. Maximum-ratio combining ([matched filtering](./Statistical_Signal_Processing.ipynb) across antennas!) collects it optimally; Alamouti's space-time code famously buys transmit diversity with two antennas and zero channel knowledge.

In [3]:
def ber_mrc(snr_db, L, n_bits=200_000):
    rho = 10**(snr_db/10)
    s = np.where(rng.random(n_bits) < 0.5, 1.0, -1.0)          # BPSK
    h = (rng.standard_normal((L, n_bits)) + 1j*rng.standard_normal((L, n_bits)))/np.sqrt(2)
    noise = (rng.standard_normal((L, n_bits)) + 1j*rng.standard_normal((L, n_bits)))/np.sqrt(2)
    y = h * s[None] + noise / np.sqrt(rho)
    s_hat = np.sign(np.real((h.conj() * y).sum(0)))            # MRC: matched filter across branches
    return np.mean(s_hat != s)

snrs = np.arange(0, 26, 5)
plt.figure(figsize=(7.5, 3))
for L in [1, 2, 4]:
    bers = [ber_mrc(s_, L) for s_ in snrs]
    plt.semilogy(snrs, bers, "o-", label=f"L={L} branches")
plt.semilogy(snrs, 0.5*10**(-snrs/10), "k:", linewidth=1, label="slope −1 reference")
plt.legend(fontsize=8); plt.xlabel("SNR [dB]"); plt.ylabel("BER")
plt.title("diversity order = the slope: each independent branch multiplies the decay rate")
plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()

/tmp/ipykernel_2992188/2963668625.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()


**What just happened.** Three BER curves, and the thing to read is not their height but their **slope**. On log-BER against dB-SNR, $L = 1$ tracks the dotted reference at roughly one decade of BER per 10 dB. $L = 2$ falls about twice as fast, and $L = 4$ about four times as fast. Diversity order *is* the slope.

That distinction matters more than it first appears. Diversity does not shift the curve downward by a fixed amount — it **tilts** it. So the benefit is not a constant number of decibels; it grows without limit as SNR rises. At 5 dB the curves are close together; by 25 dB they are separated by orders of magnitude. Any technique that changes a slope beats any technique that changes an offset, given enough SNR.

**Why fading is so punishing in the first place.** On an AWGN channel BER falls *exponentially* with SNR. On a Rayleigh channel it falls only as $1/\rho$ — one decade per 10 dB — because the error rate is dominated not by typical conditions but by the probability of a deep fade. The average SNR can be perfectly healthy while the link is unusable, and pouring in transmit power is a poor remedy: 10 dB more power buys a single decade.

Diversity attacks the actual cause. With $L$ independent branches, a deep fade requires *all* of them to fade at once, and independent rare events multiply: $P \sim \rho^{-L}$. You are not making any branch better — you are making simultaneous failure rarer.

**The word "independent" is load-bearing.** Diversity order $L$ requires the branches to fade independently. Antennas spaced much closer than half a wavelength see strongly correlated channels and deliver far less than the nominal order; two perfectly correlated branches give array gain — a downward shift — but *no* change in slope, because they always fade together. This is the entire reason antenna-spacing rules exist in real designs, and it is why the theoretical curves above are an upper bound on what hardware achieves.

**One line does the combining, and it is a familiar one.** `(h.conj() * y).sum(0)` is maximum-ratio combining: weight each branch by its own conjugate channel, so strong branches count more and weak ones are attenuated rather than discarded. That is the [matched filter](./Statistical_Signal_Processing.ipynb) applied across antennas instead of across time, and it is provably the optimal linear combiner in white noise.

**And the practical caveat.** MRC needs several *receive* antennas and channel knowledge at the receiver — fine for a base station, awkward for a handset. Alamouti's space-time block code obtains full transmit diversity from two *transmit* antennas with no channel knowledge at the transmitter at all, which is why it appears in essentially every cellular standard: it moves the antenna cost to the tower.

---
### 🕐 Session 3 of 3 — *SVD Precoding: the Channel, Diagonalized* (~40 min)
**Goal:** with channel knowledge, transform MIMO into independent parallel pipes — verified.
**Builds on:** Session 2; [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S4.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: SVD Precoding — the Channel, Diagonalized</b></summary>

**Timing (~40 min).** 10 min the diagonalisation argument · 10 min the two oracles · 12 min water-filling · 8 min the practical cost of channel knowledge.

**Board first — three lines and the whole idea is there.** Write $H = U\Sigma V^H$. Precode with $V$, receive-combine with $U^H$, and the end-to-end channel is $U^H H V = \Sigma$ — diagonal. Four antennas become four *independent scalar channels* with gains $\sigma_i$, no cross-talk, no equalisation, no interference cancellation. This is the [SVD](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) cashing its largest practical cheque in the whole curriculum, and it is worth saying so.

**Distinguish the two oracles — they check different things.** The first is *algebraic*: $\|U^HHV - \mathrm{diag}(\sigma)\|_\infty = 1.3\times10^{-15}$, which is linear algebra verified at machine precision and would hold with no data at all. The second is *statistical*: actual QPSK streams pushed through the actual channel, with cross-talk correlations measured at the 0.01 level. Students tend to see two similar-looking checks; make clear that one confirms the mathematics and the other confirms the mathematics survives contact with noise and finite records.

**Point at the 0.01 rather than calling it small.** With 20,000 symbols, the sampling noise floor for a correlation estimate is $1/\sqrt{20000} \approx 0.007$. The measured cross-talks — 0.0117, 0.0055, 0.0074, 0.0127 — are all sitting *at* that floor, so they are statistically indistinguishable from zero. That is the honest way to report a residual, and it is a much better answer than "close enough" if a student asks how close is close.

**Note stream 3's own-correlation of 0.989.** Every other stream is 1.000. The weakest pipe has $\sigma = 0.33$ against 2.91 for the strongest — nearly 9× less amplitude, so 19 dB less SNR at the same noise level. Nothing is wrong; the pipes genuinely differ in quality, which is precisely the fact water-filling exists to exploit.

**Water-filling as a decision, not a formula.** Pour power like water into a landscape whose depth is $1/\text{gain}$: strong pipes are deep and take more, weak ones may sit above the waterline and get **nothing**. Ask the room to predict what happens to the weakest pipe before running — most expect "a little"; it gets exactly zero. Then the reframe worth stating: sometimes the optimal amount of a resource to give a channel is none, and refusing to use available hardware can be the right call.

**Be careful about the +23%.** It is measured at low SNR (`P_total = N0 = 1`), and the comment in the cell says why: at high SNR water-filling converges to equal power and the gain vanishes. So 23% is not a general figure for MIMO precoding — it is the gain in the regime where power is scarce. Quoting it out of context would be wrong, and the honest version is "water-filling matters when power is tight."

**Close on the catch.** All of this requires $H$ at the *transmitter*. Getting it there means pilots, estimation, and feedback — overhead that grows with antenna count and goes stale as the channel changes, which is why this works well for slow-moving indoor Wi-Fi and poorly at vehicular speeds. Session 2's diversity methods need much less knowledge and are correspondingly more robust. That trade — performance against required channel knowledge — is the honest summary of the workshop.
</details>

## 4. The SVD Cashes Its Biggest Check

💡 **Intuition.** Write $H = U\Sigma V^H$. Precode with $V$ at the transmitter and receive with $U^H$: the end-to-end channel becomes $U^H H V = \Sigma$ — **diagonal**. Four antennas, four *independent* scalar channels with gains $\sigma_i$, zero cross-talk, no equalization. Then pour power by [water-filling](../Intro_Math/Information_Theory/Information_Theory.ipynb): more into strong pipes, none into hopeless ones. This is exactly how LTE/5G/Wi-Fi beamforming works when the channel is known.

In [4]:
nt = nr = 4
H = rayleigh(nr, nt)
U, s_vals, Vh = np.linalg.svd(H)

n_sym = 20_000
X = np.stack([qpsk_syms(n_sym) for _ in range(nt)])
tx = Vh.conj().T @ X                                        # precode with V
noise = (rng.standard_normal((nr, n_sym)) + 1j*rng.standard_normal((nr, n_sym)))/np.sqrt(2) * 0.05
Y = H @ tx + noise
Z = U.conj().T @ Y                                          # receive-combine with U^H

# ORACLE 1: effective channel is diagonal with the singular values
H_eff = U.conj().T @ H @ Vh.conj().T
off_diag = np.abs(H_eff - np.diag(s_vals)).max()
print(f"‖UᴴHV − diag(σ)‖∞ = {off_diag:.2e}   singular values: {s_vals.round(3)}")
assert off_diag < 1e-12

# ORACLE 2: the pipes are independent — cross-talk between streams ≈ 0
for i in range(nt):
    others = [j for j in range(nt) if j != i]
    xcorr = max(abs(np.corrcoef(np.real(Z[i]), np.real(X[j]))[0,1]) for j in others)
    print(f"stream {i}: gain σ={s_vals[i]:.2f}, own-corr {abs(np.corrcoef(np.real(Z[i]), np.real(X[i]))[0,1]):.3f}, "
          f"max cross-talk corr {xcorr:.4f}")

‖UᴴHV − diag(σ)‖∞ = 1.28e-15   singular values: [2.912 2.226 1.431 0.327]
stream 0: gain σ=2.91, own-corr 1.000, max cross-talk corr 0.0117
stream 1: gain σ=2.23, own-corr 1.000, max cross-talk corr 0.0055
stream 2: gain σ=1.43, own-corr 0.999, max cross-talk corr 0.0074
stream 3: gain σ=0.33, own-corr 0.989, max cross-talk corr 0.0127


**What just happened.** Two checks, testing genuinely different claims.

**The algebraic oracle.** $\|U^HHV - \mathrm{diag}(\sigma)\|_\infty = 1.3\times10^{-15}$ — machine precision. Precoding with $V$ and combining with $U^H$ turns the $4\times4$ channel into a diagonal matrix, exactly. Four antennas have become **four independent scalar channels** with gains $[2.912, 2.226, 1.431, 0.327]$: no cross-talk, no equalisation, no interference cancellation, just four separate one-dimensional links sharing a band. This is the [SVD](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) doing the most practically valuable thing it does anywhere in this curriculum.

**The statistical oracle.** Algebra is one thing; real QPSK streams through a noisy channel are another. Each stream's own-correlation is essentially 1 and the maximum cross-talk correlation is around 0.01 — the pipes really are independent when driven with data.

Read that 0.01 properly rather than calling it "small". With 20,000 symbols, the sampling noise floor for a correlation estimate is $1/\sqrt{20000} \approx 0.007$. The measured values — 0.0117, 0.0055, 0.0074, 0.0127 — all sit *at* that floor, so they are statistically indistinguishable from zero. The residual is finite-sample noise, not leakage.

**Stream 3 is the informative row.** Its own-correlation is 0.989 where the others are 1.000, because its gain is $\sigma = 0.33$ against 2.91 for the strongest — nearly 9× less amplitude, so about 19 dB worse SNR at the same noise level. Nothing has gone wrong. The pipes genuinely differ in quality, sometimes by an order of magnitude, and that spread is not a defect to be fixed but a *resource allocation problem* to be solved. The next cell solves it.

**What it costs.** Everything here assumed $H$ known at both ends. The receiver can estimate it from pilots, but the *transmitter* needs it too — and getting it there means feedback, which costs overhead that scales with antenna count and goes stale as the channel changes. That is why this works beautifully for a stationary laptop on Wi-Fi and poorly at vehicular speeds, and why Session 2's diversity schemes, which need far less knowledge, remain the robust fallback. The workshop's real summary is a trade: more channel knowledge buys more performance, and knowledge is not free.

In [5]:
# water-filling on the four pipes — at LOW SNR, where the choice matters
# (at high SNR water-filling ≈ equal power; the gains appear when power is scarce)
P_total, N0 = 1.0, 1.0
gains = s_vals**2 / N0
def waterfill(gains, P):
    mu_lo, mu_hi = 0, 1e6
    for _ in range(100):
        mu = (mu_lo + mu_hi)/2
        p = np.maximum(mu - 1/gains, 0)
        if p.sum() > P: mu_hi = mu
        else: mu_lo = mu
    return np.maximum(mu - 1/gains, 0)
p_wf = waterfill(gains, P_total)
rate_wf = np.log2(1 + gains*p_wf).sum()
rate_eq = np.log2(1 + gains*P_total/4).sum()
print(f"pipe gains σ²/N0: {gains.round(2)}")
print(f"power allocation (water-filling): {p_wf.round(2)}  — the weakest pipe gets {'NOTHING' if p_wf.min() < 1e-3 else 'little'}")
print(f"rate: equal power {rate_eq:.2f} b/s/Hz   water-filling {rate_wf:.2f} b/s/Hz  (+{100*(rate_wf/rate_eq-1):.0f}%)")

pipe gains σ²/N0: [8.48 4.95 2.05 0.11]
power allocation (water-filling): [0.48 0.4  0.11 0.  ]  — the weakest pipe gets NOTHING
rate: equal power 3.44 b/s/Hz   water-filling 4.24 b/s/Hz  (+23%)


**What just happened.** Four pipes with gains $[8.48, 4.95, 2.05, 0.11]$ — a spread of nearly 80× between best and worst — and water-filling allocated power $[0.48, 0.40, 0.11, 0.00]$. The weakest pipe receives **exactly nothing**. Rate rises from 3.44 to 4.24 bits/s/Hz, a 23% gain, purely from deciding where to put power that was already available.

**The zero is the interesting entry.** Most people predict the weak pipe gets "a little"; the optimum is none at all. Water-filling pours power into a landscape whose depth is $1/\text{gain}$, and the fourth pipe is so shallow that it sits *above* the waterline — spending any power there would buy less rate than the same power spent elsewhere. The reframe worth taking away: sometimes the optimal allocation to a channel is zero, and declining to use hardware you already own is the correct engineering decision rather than waste.

Notice this is the exact opposite of the intuition that drives diversity in Session 2. There, every branch was valuable because you were insuring against fades. Here, with channel knowledge in hand, you already *know* which pipe is bad — so there is nothing to insure against and no reason to feed it.

**Be careful how you quote the 23%.** The comment in the cell flags the condition and it matters: this is measured at **low SNR**, with `P_total = N0 = 1`. When power is scarce, concentrating it matters a great deal. At high SNR the water level rises well above every pipe's floor, water-filling converges toward equal power, and the advantage shrinks toward nothing. So 23% is not a general figure for MIMO precoding — it is the gain in the regime where power is the binding constraint. The honest statement is directional: *water-filling matters when power is tight, and stops mattering when it is plentiful*.

**Why the algorithm is a bisection.** `waterfill` searches for the water level $\mu$ such that total allocated power equals the budget. The allocation $p_i = \max(\mu - 1/g_i, 0)$ is monotone in $\mu$, so bisection converges reliably — a clean example of a constrained optimisation with an interpretable Lagrange multiplier, since $\mu$ *is* the water level and the KKT conditions are visible as the $\max(\cdot, 0)$.

**Bringing the three sessions together.** Scattering builds rank; rank builds pipes (Session 1). Independent branches steepen the BER slope when you lack channel knowledge (Session 2). With channel knowledge, the SVD diagonalises the link and water-filling optimally feeds the resulting pipes (Session 3). The through-line is that each step buys performance with a different currency — environment, antennas, or knowledge — and knowing which you can afford is the actual engineering.

## 5. Conclusion

Scattering builds rank, rank builds pipes; independent branches steepen the BER slope (measured); and the SVD — with channel knowledge — diagonalizes the link into verified-independent scalar channels fed by water-filling. The [SVD](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) never worked harder.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — the same antennas, pointed at *directions* instead of *rank*.
- [Channel Coding](./Channel_Coding.ipynb) — the codes riding inside each pipe.